In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import joblib
import os

# ==========================================
# 1. LOAD DATA
# ==========================================
print("Loading data...")
# Load your optimized dataset
model_data = pd.read_parquet("../data/processed/preprocessed_data_enriched.parquet")
model_data = model_data.sort_values(by=["item_id", "date"]).reset_index(drop=True)

# Create "Time Dummy" directly on the main dataframe for Linear Regression
model_data["time_step"] = (model_data["date"] - model_data["date"].min()).dt.days

# ==========================================
# 2. DEFINE THE SPLIT MASKS (NO COPIES!)
# ==========================================
max_date = model_data["date"].max()
split_date = max_date - pd.Timedelta(days=28)

print(f"Dataset ranges from {model_data['date'].min().date()} to {max_date.date()}")
print(f"Validation starts on: {split_date.date()}")

train_mask = model_data["date"] < split_date
val_mask = model_data["date"] >= split_date

# ==========================================
# 3. DIRECT XGBOOST (Tweedie Loss)
# ==========================================
print("Training Direct XGBoost with Tweedie Loss (Kaggle Architecture)...")

# Drop time markers and the time_step we no longer need
drop_cols = ["id", "d", "date", "sales", "wm_yr_wk", "time_step"]
features = [col for col in model_data.columns if col not in drop_cols]

# Pull X and y slices on the fly
X_train_xgb = model_data.loc[train_mask, features]
y_train_xgb = model_data.loc[train_mask, "sales"]

X_val_xgb = model_data.loc[val_mask, features]
y_val_xgb = model_data.loc[val_mask, "sales"]

# Initialize Kaggle-Tuned XGBoost Regressor
xgb_model = xgb.XGBRegressor(
    objective='reg:tweedie', 
    tweedie_variance_power=1.1, # The magic number for retail data
    eval_metric='rmse',
    n_estimators=2000,          # Increased trees because learning rate is slower
    learning_rate=0.03,         # Slower learning for better accuracy
    subsample=0.75,
    colsample_bytree=0.75,
    max_depth=8,
    min_child_weight=50,        # Prevents overfitting on sparse items
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50
)

# Fit the XGBoost model
xgb_model.fit(
    X_train_xgb, y_train_xgb,
    eval_set=[(X_train_xgb, y_train_xgb), (X_val_xgb, y_val_xgb)],
    verbose=50
)

# ==========================================
# 4. EVALUATION
# ==========================================
print("Calculating Predictions...")
final_xgb_pred = xgb_model.predict(X_val_xgb)

# Floor predictions at 0 (Tweedie naturally does this, but good safety measure)
final_xgb_pred = np.clip(final_xgb_pred, a_min=0, a_max=None)

rmse = np.sqrt(mean_squared_error(y_val_xgb, final_xgb_pred))
absolute_errors = np.abs(y_val_xgb - final_xgb_pred)
wmape = np.sum(absolute_errors) / np.sum(y_val_xgb)
business_accuracy = (1 - wmape) * 100

print("\n--- Kaggle-Tuned XGBoost Performance ---")
print(f"Validation RMSE:   {rmse:.4f}")
print(f"WMAPE:             {wmape:.4f} ({wmape * 100:.2f}%)")
print(f"Business Accuracy: {business_accuracy:.2f}%")

# ==========================================
# 5. SAVE MODEL
# ==========================================
os.makedirs("../models", exist_ok=True)
joblib.dump(xgb_model, "../models/xgboost_direct_tweedie.pkl")
print("\nSaved Direct XGBoost model successfully!")

Loading data...
Dataset ranges from 2011-02-26 to 2016-04-24
Validation starts on: 2016-03-27
Training Direct XGBoost with Tweedie Loss (Kaggle Architecture)...
[0]	validation_0-rmse:4.42764	validation_1-rmse:3.46688
[50]	validation_0-rmse:2.57881	validation_1-rmse:2.11972
[100]	validation_0-rmse:2.33715	validation_1-rmse:2.01343
[150]	validation_0-rmse:2.30877	validation_1-rmse:2.01235
[169]	validation_0-rmse:2.30372	validation_1-rmse:2.01246
Calculating Predictions...

--- Kaggle-Tuned XGBoost Performance ---
Validation RMSE:   2.0120
WMAPE:             0.6902 (69.02%)
Business Accuracy: 30.98%

Saved Direct XGBoost model successfully!
